# Error analysis
Reads `results/` only.

In [1]:
import os, sys
from pathlib import Path
ROOT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
os.chdir(ROOT); sys.path.insert(0, str(ROOT))
import pandas as pd
pd.set_option("display.width", 160)

In [2]:
s = pd.read_csv("results/summary.csv")
main = s[(s.exp == "main") & (s.train.str.len() == 1)]
main.pivot_table(index=["detector", "train"], columns="test", values="f1_mean").round(3)

test                                  A      B      C      D      E
detector                   train                                   
coord_minus_lexical_lr     A      0.996  0.998  0.998  0.639  0.997
                           B      0.991  1.000  1.000  0.933  1.000
                           C      0.985  0.994  1.000  0.557  0.978
coord_minus_lexical_svm    A      0.994  0.989  0.994  0.645  0.989
                           B      0.974  0.990  0.990  0.930  0.990
                           C      0.983  0.994  1.000  0.603  0.983
coord_minus_llm_judge      A      0.996  0.998  0.998  0.660  0.997
                           B      0.994  1.000  1.000  0.933  1.000
                           C      0.983  0.994  1.000  0.578  0.978
coord_minus_stylometry_xgb A      0.996  0.998  0.998  0.690  1.000
                           B      0.994  0.996  0.996  0.895  0.997
                           C      0.985  0.996  1.000  0.502  0.978
coord_minus_transformer    A      0.998  0.998  0.998  0.373  0.957
                           B      0.970  0.998  0.994  0.836  0.984
                           C      0.908  0.965  0.996  0.202  0.947
coordinator                A      0.996  0.998  0.998  0.639  0.997
                           B      0.994  1.000  1.000  0.933  1.000
                           C      0.980  0.994  1.000  0.557  0.978
lexical_lr                 A      0.994  0.996  0.996  0.304  0.990
                           B      0.981  0.998  0.996  0.571  0.993
                           C      0.897  0.933  0.998  0.118  0.918
lexical_svm                A      0.996  0.996  0.998  0.233  0.983
                           B      0.989  1.000  0.998  0.523  0.993
                           C      0.925  0.948  0.998  0.118  0.939
llm_judge                  A      0.494  0.428  0.589  0.541  0.626
                           B      0.494  0.428  0.589  0.541  0.626
                           C      0.494  0.428  0.589  0.541  0.626
stylometry_xgb             A      0.937  0.864  0.858  0.292  0.434
                           B      0.865  0.917  0.896  0.906  0.937
                           C      0.831  0.896  0.947  0.827  0.915
transformer                A      0.985  0.985  0.985  0.841  0.980
                           B      0.977  0.980  0.980  0.932  0.973
                           C      0.983  0.989  0.991  0.659  0.970

## Missed LLM texts on the external generator D (Coordinator trained on A)

In [3]:
p = pd.concat(pd.read_parquet(f).assign(seed=f.stem[-4:]) for f in Path("results/preds").glob("main__train-A__*.parquet"))
ds = pd.read_parquet("data/processed/dataset.parquet")[["id", "text", "n_words"]]
fn = p[(p.test == "D") & (p.label == 1) & (p.coordinator < 0.5)].merge(ds, on="id")
print(len(fn), "false negatives")
for _, r in fn.drop_duplicates("id").head(3).iterrows():
    print(f"--- p={r.coordinator:.2f} lexical={r.lexical_lr:.2f} roberta={r.transformer:.2f}\n{r.text[:500]}\n")

23 false negatives
--- p=0.26 lexical=0.41 roberta=0.98
Network defenders are being urged to heighten vigilance against a resurgence of exploitation activity targeting the widely known Citrix Application Delivery Controller (ADC) and Citrix Gateway vulnerability identified as CVE-2019-19781. Intelligence reports indicate that a range of sophisticated threat actors-including state-aligned advanced persistent threat (APT) groups and other malicious cyber actors-have revived this exploit to gain unauthorized access to internal networks across a broad sp

--- p=0.00 lexical=0.26 roberta=0.00
The investigation began after multiple unrelated alerts were correlated across several enterprise networks. Log-source analysis showed that malicious e-mail messages containing either attachments or links to publicly accessible file-hosting services were the common infection vector. The attachments were typically Microsoft Office documents or PDFs that, when opened, dropped one of four remote-access to

## Recall by topic and template

In [4]:
pd.read_csv("results/recall_by_topic.csv")

,setting,topic,n,recall,recall_std
0,cross,apt,537,0.899882,0.231375
1,cross,malware,87,0.988889,0.060858
2,cross,other,48,1.000000,0.000000
3,cross,phishing,312,0.822045,0.333603
4,cross,ransomware,471,0.903419,0.258635
5,cross,vulnerability,522,0.885673,0.213779
6,in,apt,201,1.000000,0.000000
7,in,malware,30,1.000000,0.000000
8,in,other,21,1.000000,0.000000
9,in,phishing,102,1.000000,0.000000


In [5]:
pd.read_csv("results/recall_by_template.csv")

,setting,template,n,recall,recall_std
0,cross,t1,445,0.907407,0.249426
1,cross,t2,455,0.861256,0.276374
2,cross,t3,520,0.922982,0.170314
3,cross,t4,557,0.888690,0.207490
4,in,t1,167,1.000000,0.000000
5,in,t2,160,1.000000,0.000000
6,in,t3,185,1.000000,0.000000
7,in,t4,199,0.994709,0.015873


## Paired bootstrap: in- vs cross-generator F1

In [6]:
h = pd.read_csv("results/h1_bootstrap.csv")
h.groupby(["detector", "test"])[["diff", "ci_low", "ci_high"]].mean().round(3)

diff  ci_low  ci_high
detector       test                        
coordinator    A     0.013   0.001    0.032
               B     0.002  -0.003    0.011
               C    -0.001  -0.003    0.000
               D     0.289   0.123    0.507
               E     0.007  -0.001    0.017
lexical_lr     A     0.059   0.026    0.101
               B     0.032   0.006    0.061
               C     0.000  -0.007    0.008
               D     0.666   0.463    0.920
               E     0.030  -0.000    0.065
lexical_svm    A     0.042   0.017    0.075
               B     0.025   0.005    0.047
               C    -0.000  -0.004    0.003
               D     0.707   0.530    0.937
               E     0.026   0.001    0.059
llm_judge      A     0.014  -0.063    0.093
               B     0.114   0.035    0.196
               C    -0.128  -0.203   -0.058
               D    -0.037  -0.167    0.218
               E    -0.122  -0.197   -0.049
stylometry_xgb A     0.084   0.029    0.141
               B     0.062   0.015    0.116
               C     0.050   0.001    0.103
               D     0.259   0.072    0.452
               E     0.172   0.092    0.256
transformer    A     0.005   0.000    0.014
               B     0.001   0.000    0.003
               C     0.000   0.000    0.000
               D     0.175   0.043    0.344
               E     0.011  -0.004    0.029